# Gout MLOps — ejecución en Colab

Colab no tiene Docker, así que acá no se levanta el `docker-compose`. Este notebook corre **el mismo código**
que usan los DAGs (`src/gout_common`), reemplazando los servicios por equivalentes locales:

| En Docker | En Colab |
|---|---|
| PostgreSQL (`gout_db`) | SQLite en `/content/gout_colab/gout.db` |
| MinIO (`raw/pedidos_anon.csv`) | CSV en Drive (`MyDrive/gout/data/`) |
| Airflow (`etl_gout`, `train_gout`) | llamadas directas a `gout_common.etl` / `gout_common.train` |
| Servidor MLflow | MLflow local (`mlflow.db` + `mlruns/` en `/content/gout_colab`) |
| Contenedores `api` y `grpc` | los mismos procesos, en segundo plano (puertos 8800 y 50051) |
| Kafka + `producer` / `consumer` | Kafka 3.8 (KRaft) descargado en Colab + los mismos scripts |
| DAG `drift_gout` (reporte en MinIO) | `gout_common.drift` (reporte HTML en `gout_colab/reportes_drift`) |

Requisitos en Drive:
- `MyDrive/gout/gout-mlops.zip` (el repo)
- `MyDrive/gout/data/pedidos_anon.csv` (salida del notebook de anonimización)

## 1. Drive y repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, zipfile
from pathlib import Path

MYDRIVE  = Path('/content/drive/MyDrive')
WORK_DIR = Path('/content/gout_colab')
REPO_DIR = Path('/content/gout-mlops')

def buscar(patron):
    # busca en todo MyDrive (cualquier subcarpeta); devuelve el más reciente
    hallados = [p for p in MYDRIVE.rglob(patron) if '/.Trash' not in str(p)]
    return max(hallados, key=lambda p: p.stat().st_mtime) if hallados else None

repo_zip = buscar('gout-mlops*.zip')
repo_dir = next((p.parent.parent for p in MYDRIVE.rglob('gout_common') if p.parent.name == 'src'), None)
if repo_zip:
    with zipfile.ZipFile(repo_zip) as z:
        z.extractall('/content')
    print('Repo desde', repo_zip)
elif repo_dir:
    REPO_DIR = repo_dir
    print('Repo desde carpeta', repo_dir)
else:
    raise FileNotFoundError('No encontré gout-mlops*.zip ni la carpeta del repo en MyDrive')

RAW_CSV = buscar('pedidos_anon.csv')
if RAW_CSV is None:
    raise FileNotFoundError('No encontré pedidos_anon.csv en MyDrive (correr antes el notebook de anonimización)')
GOUT_DIR = RAW_CSV.parent.parent          # carpeta donde se guarda la base al final
print('CSV desde', RAW_CSV)
WORK_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo en', REPO_DIR)

## 2. Dependencias
Mismas versiones que las imágenes Docker, para que el modelo sea compatible con el serving. SQLAlchemy < 2.1 porque MLflow 2.22 no funciona con 2.1.

In [ ]:
!pip install -q xgboost==2.1.3 scikit-learn==1.5.2 mlflow==2.22.1 "sqlalchemy>=2.0,<2.1" \
    fastapi==0.142.2 "uvicorn[standard]==0.54.0" "strawberry-graphql[fastapi]==0.327.7" graphql-core==3.2.6 \
    grpcio-tools==1.68.1 confluent-kafka==2.6.1 requests

## 3. Configuración
Las variables de entorno se fijan **antes** de importar `gout_common` (config las lee al importar).

In [ ]:
os.environ['GOUT_DB_URL'] = f'sqlite:///{WORK_DIR}/gout.db'
os.environ['TEST_YEAR']   = '2025'
os.environ['MLFLOW_TRACKING_URI'] = f'sqlite:///{WORK_DIR}/mlflow.db'
os.chdir(WORK_DIR)   # los artefactos de MLflow quedan en WORK_DIR/mlruns

import pandas as pd
from gout_common import config, etl

engine = etl.get_engine()
print(config.GOUT_DB_URL)

## 4. ETL (mismas tareas que el DAG `etl_gout`)

In [ ]:
# extraer: en Docker lee de MinIO; acá el CSV de Drive
stats_extraer = etl.run_extract(engine, raw=pd.read_csv(RAW_CSV))
stats_extraer

In [ ]:
stats_externas = etl.run_externals(engine)
stats_externas   # clima_real=True si Open-Meteo respondió

In [ ]:
stats_features = etl.run_features(engine)
stats_features

## 5. Verificación

In [ ]:
from sqlalchemy import inspect
for t in inspect(engine).get_table_names():
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', engine)['n'][0]
    print(f'{t:<20} {n:>8,}')

train = pd.read_sql(f'SELECT * FROM {config.TABLE_TRAIN}', engine, parse_dates=['FECHA'])
test  = pd.read_sql(f'SELECT * FROM {config.TABLE_TEST}',  engine, parse_dates=['FECHA'])
print(f"\nTrain: {train['FECHA'].min().date()} → {train['FECHA'].max().date()}")
print(f"Test : {test['FECHA'].min().date()} → {test['FECHA'].max().date()}")
train.head()

## 6. Entrenamiento (mismas tareas que el DAG `train_gout`)
`run_train` entrena el pipeline de la memoria y lo registra en MLflow; `run_promote` lo pasa a `prod`
solo si mejora el RMSE del modelo vigente sobre el mismo test.

In [ ]:
import logging
logging.getLogger('mlflow').setLevel(logging.ERROR)
from gout_common import train as tr

resultado = tr.run_train(engine)
{k: v for k, v in resultado.items() if k != 'run_id'}

In [ ]:
decision = tr.run_promote(engine, resultado['version'])
decision

In [ ]:
# Registry: versiones, alias y estado
from mlflow import MlflowClient
client = MlflowClient()
filas = []
for v in client.search_model_versions(f"name='{config.MODEL_NAME}'"):
    v = client.get_model_version(config.MODEL_NAME, v.version)   # search no trae los alias
    filas.append({'version': v.version, 'aliases': v.aliases,
                  'estado': v.tags.get('estado'), 'test_rmse': v.tags.get('test_rmse')})
pd.DataFrame(filas).sort_values('version')

In [ ]:
# Artefactos del run: métricas por sucursal e importancia de atributos
import mlflow
d = mlflow.artifacts.download_artifacts(run_id=resultado['run_id'], artifact_path='evaluacion')
display(pd.read_csv(f'{d}/metricas_por_sucursal.csv').round(3))
pd.read_csv(f'{d}/importancia_atributos.csv').round(3)

In [ ]:
# Prueba del modelo en prod tal como lo carga el serving
from gout_common.model import reload_model
from gout_common.features import FEATURES
modelo = reload_model()
muestra = test.head(5)
muestra[['FECHA', 'SUCURSAL', 'SKU', 'TOTAL']].assign(PRED=modelo.predict(muestra[FEATURES]).round(2))

## 7. Serving: REST, GraphQL y gRPC
Se levantan la API (REST + GraphQL) y el servidor gRPC como procesos en segundo plano, con el mismo código
que los contenedores. Los tres usan `gout_common.serving`, así que deben devolver lo mismo.

In [ ]:
import subprocess, time, requests

env = {**os.environ, 'PYTHONPATH': f"{REPO_DIR / 'src'}:{REPO_DIR / 'api'}:{REPO_DIR}", 'GRPC_PORT': '50051'}

# Código gRPC generado a partir del .proto (en Docker lo hace el Dockerfile)
subprocess.run([sys.executable, '-m', 'grpc_tools.protoc', '-I', 'grpc_service/protos',
                '--python_out=grpc_service', '--grpc_python_out=grpc_service',
                'grpc_service/protos/forecast.proto'], cwd=REPO_DIR, check=True)
pb2g = REPO_DIR / 'grpc_service' / 'forecast_pb2_grpc.py'
pb2g.write_text(pb2g.read_text().replace('import forecast_pb2', 'from grpc_service import forecast_pb2'))

for p in globals().get('procesos', []):
    p.terminate()
procesos = [
    subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app.main:app', '--port', '8800'],
                     cwd=REPO_DIR / 'api', env=env, stdout=open(WORK_DIR / 'api.log', 'w'), stderr=subprocess.STDOUT),
    subprocess.Popen([sys.executable, '-m', 'grpc_service.server'],
                     cwd=REPO_DIR, env=env, stdout=open(WORK_DIR / 'grpc.log', 'w'), stderr=subprocess.STDOUT),
]

API = 'http://localhost:8800'
for _ in range(60):
    try:
        if requests.get(f'{API}/health', timeout=2).json().get('status') == 'ok':
            break
    except Exception:
        pass
    time.sleep(1)
print(requests.get(f'{API}/health').json())
print('Logs en', WORK_DIR / 'api.log', 'y', WORK_DIR / 'grpc.log')

In [ ]:
import grpc
sys.path.insert(0, str(REPO_DIR))
from grpc_service import forecast_pb2 as pb, forecast_pb2_grpc as pb_grpc
stub = pb_grpc.ForecastStub(grpc.insecure_channel('localhost:50051'))

# Una fila real de 2025: los tres protocolos deben coincidir con el pipeline
r = test.sample(1, random_state=7).iloc[0]
f = str(r['FECHA'].date())
pedido = dict(fecha=f, sucursal=r['SUCURSAL'], sku=r['SKU'], stock=int(r['STOCK']), devoluciones=int(r['DEVOLUCIONES']))

rest = requests.post(f'{API}/predict', json=pedido).json()
q = '{ pronostico(sucursal:"%s", fecha:"%s", items:[{sku:"%s", stock:%d, devoluciones:%d}]) { modeloVersion pronosticos { sku unidades } } }' % (
    r['SUCURSAL'], f, r['SKU'], r['STOCK'], r['DEVOLUCIONES'])
gql = requests.post(f'{API}/graphql', json={'query': q}).json()
grp = stub.Predict(pb.PredictRequest(**pedido))

print('Pedido  :', pedido, '| real:', int(r['TOTAL']))
print('REST    :', rest)
print('GraphQL :', gql['data'])
print('gRPC    :', grp.unidades, '(versión', grp.modelo_version + ')')

In [ ]:
# Pronóstico de una sucursal completa para una fecha (batch)
dia = test[(test['FECHA'] == test['FECHA'].max()) & (test['SUCURSAL'] == 'S01')]
items = [dict(sku=s, stock=int(k), devoluciones=int(d)) for s, k, d in zip(dia['SKU'], dia['STOCK'], dia['DEVOLUCIONES'])]
batch = requests.post(f'{API}/predict/batch', json=dict(fecha=str(dia['FECHA'].iloc[0].date()), sucursal='S01', items=items)).json()
pd.DataFrame(batch['predicciones']).assign(real=dia['TOTAL'].values)[['sku', 'unidades', 'real']]

In [ ]:
# Comparación de latencia entre protocolos (misma lógica, distinto transporte)
import statistics
filas = test.sample(200, random_state=0)
pedidos = [dict(fecha=str(x.FECHA.date()), sucursal=x.SUCURSAL, sku=x.SKU, stock=int(x.STOCK), devoluciones=int(x.DEVOLUCIONES))
           for x in filas.itertuples()]

def medir(fn):
    ms = []
    for p in pedidos:
        t = time.perf_counter(); fn(p); ms.append((time.perf_counter() - t) * 1000)
    ms.sort()
    return {'media_ms': round(statistics.mean(ms), 1), 'p50_ms': round(ms[len(ms)//2], 1), 'p95_ms': round(ms[int(len(ms)*.95)], 1)}

sesion = requests.Session()
q_tpl = '{ pronostico(sucursal:"%s", fecha:"%s", items:[{sku:"%s", stock:%d, devoluciones:%d}]) { pronosticos { unidades } } }'
latencias = pd.DataFrame({
    'REST':    medir(lambda p: sesion.post(f'{API}/predict', json=p).json()),
    'GraphQL': medir(lambda p: sesion.post(f'{API}/graphql', json={'query': q_tpl % (p['sucursal'], p['fecha'], p['sku'], p['stock'], p['devoluciones'])}).json()),
    'gRPC':    medir(lambda p: stub.Predict(pb.PredictRequest(**p))),
}).T
latencias

In [ ]:
# Cada predicción queda registrada (insumo del monitoreo de drift)
pd.read_sql(f'SELECT "PROTOCOLO", COUNT(*) AS n, AVG("MS_MODELO") AS ms_modelo FROM {config.TABLE_PREDICTIONS} GROUP BY "PROTOCOLO"', engine)

## 8. Streaming con Kafka (S4)
El productor re-emite los pedidos reales de un período, fecha a fecha. El consumidor pide cada pronóstico
por gRPC y, si el pedido se desvía del pronóstico más que el umbral (`max(4 u., 50 %)`), publica una alerta
de **pedido atípico** (SOBREPEDIDO / SUBPEDIDO) en el tópico `alertas_desvio`. Todas las comparaciones
quedan en `evaluacion_online` (insumo del monitoreo).

In [ ]:
# Kafka 3.8 en modo KRaft (sin ZooKeeper), igual que la imagen apache/kafka del docker-compose
KAFKA_DIR = Path('/content/kafka_2.13-3.8.0')
if not KAFKA_DIR.exists():
    subprocess.run('wget -q https://archive.apache.org/dist/kafka/3.8.0/kafka_2.13-3.8.0.tgz -O /content/kafka.tgz '
                   '&& tar -xzf /content/kafka.tgz -C /content', shell=True, check=True)
if subprocess.run('java -version', shell=True, capture_output=True).returncode != 0:
    subprocess.run('apt-get install -y -qq openjdk-17-jre-headless > /dev/null', shell=True, check=True)

import socket
def kafka_arriba():
    with socket.socket() as s:
        return s.connect_ex(('localhost', 9092)) == 0

if not kafka_arriba():
    cfg = KAFKA_DIR / 'config' / 'kraft' / 'server.properties'
    uuid = subprocess.run([KAFKA_DIR / 'bin' / 'kafka-storage.sh', 'random-uuid'], capture_output=True, text=True).stdout.strip()
    subprocess.run([KAFKA_DIR / 'bin' / 'kafka-storage.sh', 'format', '-t', uuid, '-c', cfg, '--ignore-formatted'],
                   check=True, capture_output=True)
    subprocess.run([KAFKA_DIR / 'bin' / 'kafka-server-start.sh', '-daemon', cfg], check=True)
    for _ in range(60):
        if kafka_arriba():
            break
        time.sleep(1)
print('Kafka arriba:', kafka_arriba())

In [ ]:
# Cliente gRPC del streaming (en Docker lo genera el Dockerfile de streaming)
subprocess.run([sys.executable, '-m', 'grpc_tools.protoc', '-I', 'grpc_service/protos',
                '--python_out=streaming', '--grpc_python_out=streaming',
                'grpc_service/protos/forecast.proto'], cwd=REPO_DIR, check=True)
pb2g = REPO_DIR / 'streaming' / 'forecast_pb2_grpc.py'
pb2g.write_text(pb2g.read_text().replace('import forecast_pb2', 'from streaming import forecast_pb2'))

env_stream = {**env, 'PYTHONPATH': str(REPO_DIR), 'KAFKA_BOOTSTRAP': 'localhost:9092', 'GRPC_TARGET': 'localhost:50051',
              'REPLAY_DESDE': '2025-06-01', 'REPLAY_HASTA': '2025-06-30', 'REPLAY_PAUSA_S': '0.5',
              'MAX_INACTIVO_S': '20'}   # el consumidor termina solo tras 20 s sin mensajes

consumidor = subprocess.Popen([sys.executable, '-m', 'streaming.consumer'], cwd=REPO_DIR, env=env_stream,
                              stdout=open(WORK_DIR / 'consumer.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
prod = subprocess.run([sys.executable, '-m', 'streaming.producer'], cwd=REPO_DIR, env=env_stream,
                      capture_output=True, text=True)
print(prod.stdout[-1500:] or prod.stderr[-1500:])
consumidor.wait(timeout=600)
print(open(WORK_DIR / 'consumer.log').read()[-1500:])

In [ ]:
online  = pd.read_sql('SELECT * FROM evaluacion_online', engine, parse_dates=['FECHA'])
alertas = pd.read_sql('SELECT * FROM alertas_desvio',   engine, parse_dates=['FECHA'])
print(f"Pedidos evaluados: {len(online):,} | alertas: {len(alertas):,} ({len(alertas)/len(online):.1%})")
print('MAE online:', round((online['PEDIDO'] - online['PRONOSTICO']).abs().mean(), 3))
display(alertas.groupby(['SUCURSAL', 'ALERTA']).size().unstack(fill_value=0))
alertas.sort_values('DESVIO', key=abs, ascending=False).head(10)[
    ['FECHA', 'SUCURSAL', 'SKU', 'STOCK', 'PEDIDO', 'PRONOSTICO', 'DESVIO', 'ALERTA']]

In [ ]:
# Las alertas también quedaron publicadas en el tópico de Kafka
from confluent_kafka import Consumer
lector = Consumer({'bootstrap.servers': 'localhost:9092', 'group.id': 'lector-colab', 'auto.offset.reset': 'earliest'})
lector.subscribe(['alertas_desvio'])
n, vacios, ejemplo = 0, 0, None
while vacios < 5:
    m = lector.poll(1.0)
    if m is None:
        vacios += 1; continue
    n += 1; ejemplo = ejemplo or m.value().decode()
lector.close()
print(f'Mensajes en alertas_desvio: {n}')
print('Ejemplo:', ejemplo)

## 9. Monitoreo de drift (S7) — mismas tareas que el DAG `drift_gout`
- **Drift de datos:** PSI por atributo entre la ventana reciente (últimos 30 días con pedidos evaluados) y una
  **referencia estacional** (filas de entrenamiento de los mismos meses). PSI < 0,1 estable · 0,1–0,2 moderado · ≥ 0,2 drift.
- **Degradación:** MAE online (pedido real vs. pronóstico, de `evaluacion_online`) contra el MAE de test del modelo en prod (+20 % de tolerancia).
- **Decisión:** reentrenar si hay drift en ≥ 30 % de los atributos o si el modelo degrada.

In [ ]:
os.environ['REPORTS_DIR'] = str(WORK_DIR / 'reportes_drift')
from gout_common import drift as dr

res = dr.run_drift(engine)
print(f"Ventana {res['desde']} → {res['hasta']} | referencia: meses {res['meses_ref']} ({res['n_ref']:,} filas)")
print('Degradación:', res['degradacion'])
print('Decisión   :', 'REENTRENAR' if res['reentrenar'] else 'sin acción', '—', res['motivo'])
res['tabla']

In [ ]:
from IPython.display import HTML
HTML(open(res['reporte'], encoding='utf-8').read())

### Escenario simulado: cambio en el patrón de pedidos
Para mostrar que el monitoreo reacciona, se perturba la ventana actual: stock +80 % y pedidos +40 %
(como si se abriera un canal nuevo de venta). No se guarda en `metricas_drift`.

In [ ]:
actual = dr.datos_ventana(engine, res['desde'], res['hasta']).copy()
actual['STOCK'] = (actual['STOCK'] * 1.8).round()
actual['TOTAL'] = (actual['TOTAL'] * 1.4).round()
for c in ['LAG_1', 'LAG_2', 'ROLLING_MEAN_3']:
    actual[c] = actual[c] * 1.4

sim = dr.run_drift(engine, actual=actual, desde=res['desde'], hasta=res['hasta'], guardar=False)
print('Decisión:', 'REENTRENAR' if sim['reentrenar'] else 'sin acción', '—', sim['motivo'])
sim['tabla']

In [ ]:
# Si la decisión es reentrenar, el DAG dispara etl_gout -> train_gout. Acá se corre el mismo circuito.
# Como los datos no cambiaron, el challenger empata con el champion y NO se promueve: el
# champion/challenger evita reemplazar el modelo en prod por uno que no es mejor.
if sim['reentrenar']:
    etl.run_features(engine)
    nuevo = tr.run_train(engine)
    tr.run_promote(engine, nuevo['version'])

In [ ]:
pd.read_sql(f'SELECT "TS", "DESDE", "HASTA", "N_DRIFT", "MAE_ONLINE", "MAE_TEST", "REENTRENAR", "MOTIVO" FROM {config.TABLE_DRIFT}', engine)

## 10. Guardar en Drive
Se copia `gout_colab/` completo (base, MLflow y artefactos). Para retomar en otra sesión, copiarlo de vuelta
a `/content/gout_colab` (las rutas de artefactos quedan registradas con esa ubicación).

In [ ]:
import shutil
destino = GOUT_DIR / 'gout_colab'
shutil.copytree(WORK_DIR, destino, dirs_exist_ok=True)
print('Guardado en', destino)